# Kitwe City Council — Multi-Source Data Extraction, Cleaning and Integration Pipeline

**Course:** CSC 4792 — Data Mining and Warehousing.
**University:** University of Zambia (UNZA), 2026
Students: Kanchele L Chipapa, Luyando T Nyirenda, Rabson Tembo, Pretorious Chikanda, Mark Mushiya
**Instructor:** Dr. Lighton Phiri (lighton.phiri@gmail.com)


Overview

This notebook documents the complete data pipeline for constructing a multi-source
dataset from **Kitwe City Council** (Copperbelt Province, Zambia) digital footprints.
The pipeline covers:

1. **Data acquisition** — downloading PDF/DOCX source documents from the Kitwe City Council website
2. **Data extraction** — scraping structured records from budget PDFs, financial statements, council minutes, and CDF project lists
3. **Data cleaning** — fixing column misalignment, resolving type mismatches, standardising identifiers
4. **Data integration** — merging two independently extracted source files into a unified schema
5. **Deduplication** — identifying and cataloguing overlapping/duplicate records
6. **Thematic splitting** — partitioning the unified dataset into thematic CSV files for Kaggle

### Legislative context

The dataset is grounded in Zambia's **Local Government Act No. 2 of 2019** and its
amendments:
- **Act 8 of 2023** — Local Government Equalisation Fund (LGEF) reform
- **Act 5 of 2026** — CDF allocation increased to K40 million per constituency









## 1. ENVIRONMENT SETUP


In [29]:
# Standard libraries
import pandas as pd
import numpy as np
import re
import os
import glob
from pathlib import Path

# PDF/DOCX extraction (uncomment if running locally)
# import pdfplumber
# import docx
# import tabula

# Display settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

print('Environment ready. pandas', pd.__version__, '| numpy', np.__version__)

Environment ready. pandas 2.2.3 | numpy 2.1.3


## 2. Data Acquisition

Source documents were obtained from the **Kitwe City Council** official website
(https://www.kitwecitycouncil.gov.zm) and the **Ministry of Local Government and Rural
Development** portal. The table below lists all source documents used.

In [30]:
# Source documents inventory
source_docs = pd.DataFrame({
    'Document': [
        'CDF-2024-PROPOSED-AND-APPROVED-PROJECTS.pdf',
        'Approved-Budget-2025.pdf',
        '2026-budget-Kitwe-City-Council-amended-Final.pdf',
        'Budget-Estimates-2023.pdf',
        'Budget-Performance-Report-116-LAs.pdf',
        '3RD-OCTOBER-2023-COUNCIL-MINUTES.docx',
        'COUNCIL-MINUTES-4TH-APRIL-2023.docx',
        '30TH-JUNE-2023-COUNCIL-MINUTES-copy.docx',
        'STAKEHOLDERS-MEETING-MINUTUES-PERSON-WITH-DISABILITIES.pdf',
        'SUMMARY-OF-ACTION-TAKEN-REPORT.pdf',
        'Kitwe-City-Council-2018-FS.pdf',
        'Kitwe-City-Council-2019-FS.pdf',
        'Kitwe-City-Council-2020-FS.pdf',
        'Kitwe-City-Council-2021-FS.pdf',
        'Kitwe-C-C-2022-Financial-Statements-SIGNED.pdf',
        'Kitwe-C-C-2023-Financial-Statements-SIGNED.pdf',
        'Kitwe-C-C-2024-Audited-financial-statements-final-071125.pdf',
        'KCC CDF Signing Jan 2026',
        'KCC CDF Contracts 2025',
        'KCC Empowerment Grants 2025',
        'Kitwe Council News'
    ],
    'Type': [
        'PDF', 'PDF', 'PDF', 'PDF', 'PDF',
        'DOCX', 'DOCX', 'DOCX', 'PDF', 'PDF',
        'PDF', 'PDF', 'PDF', 'PDF', 'PDF',
        'PDF', 'PDF',
        'Image/Document', 'Document', 'Document', 'Web Page'
    ],
    'Content': [
        'CDF proposed and approved projects for 2024',
        'Approved Budget estimates for 2025',
        'Amended Budget estimates for 2026',
        'Budget Estimates for 2023',
        'Budget performance report for 116 Local Authorities',
        'Council meeting minutes (3 Oct 2023)',
        'Council meeting minutes (4 Apr 2023)',
        'Council meeting minutes (30 Jun 2023)',
        'Stakeholders meeting minutes (persons with disabilities)',
        'Summary of Action Taken report',
        'Audited financial statements FY2018',
        'Audited financial statements FY2019',
        'Audited financial statements FY2020',
        'Audited financial statements FY2021',
        'Audited financial statements FY2022',
        'Audited financial statements FY2023',
        'Audited financial statements FY2024',
        'CDF contract signing records (Jan 2026)',
        'CDF contract award records (2025)',
        'CDF empowerment grant allocations (2025)',
        'Council news/announcements'
    ],
    'Fiscal Years': [
        '2024', '2025', '2026', '2023', '2025',
        '2023', '2023', '2023', '2025', '2023',
        '2018', '2019', '2020', '2021', '2022',
        '2023', '2024', '2026', '2025', '2025', '2023'
    ]
})

print(f'Total source documents: {len(source_docs)}')
source_docs

Total source documents: 21


,Document,Type,Content,Fiscal Years
0,CDF-2024-PROPOSED-AND-APPROVED-PROJECTS.pdf,PDF,CDF proposed and approved projects for 2024,2024
1,Approved-Budget-2025.pdf,PDF,Approved Budget estimates for 2025,2025
2,2026-budget-Kitwe-City-Council-amended-Final.pdf,PDF,Amended Budget estimates for 2026,2026
3,Budget-Estimates-2023.pdf,PDF,Budget Estimates for 2023,2023
4,Budget-Performance-Report-116-LAs.pdf,PDF,Budget performance report for 116 Local Authorities,2025
5,3RD-OCTOBER-2023-COUNCIL-MINUTES.docx,DOCX,Council meeting minutes (3 Oct 2023),2023
6,COUNCIL-MINUTES-4TH-APRIL-2023.docx,DOCX,Council meeting minutes (4 Apr 2023),2023
7,30TH-JUNE-2023-COUNCIL-MINUTES-copy.docx,DOCX,Council meeting minutes (30 Jun 2023),2023
8,STAKEHOLDERS-MEETING-MINUTUES-PERSON-WITH-DISABILITIES.pdf,PDF,Stakeholders meeting minutes (persons with disabilities),2025
9,SUMMARY-OF-ACTION-TAKEN-REPORT.pdf,PDF,Summary of Action Taken report,2023


## 3. Data Extraction

Data was extracted from source documents using two complementary approaches:

- **File 1** (`kitwe_city_council_data__1_(1).csv`): Manually curated from **audited
  financial statements** and **budget performance reports** — these contain audited
  actual figures verified by the Auditor General's office.

- **File 2** (`DataMining-project.csv`): Data-mined from **council minutes**,
  **CDF project lists**, and **budget estimate PDFs** using programmatic extraction
  (pdfplumber, tabula-py, python-docx).

The two files use different column schemas and have a partially overlapping row
space, requiring careful schema alignment and deduplication.

In [47]:
# Load both source CSVs (all columns as string to preserve raw data)
f1_raw = pd.read_csv('/kitwe_city_council_data__1_(1).csv', dtype=str)
f2_raw = pd.read_csv('/DataMining-project.csv', dtype=str)

print(f'File 1 (audited actuals): {f1_raw.shape[0]} rows, {f1_raw.shape[1]} columns')
print(f'Columns: {list(f1_raw.columns)}')
print()
print(f'File 2 (data-mined):      {f2_raw.shape[0]} rows, {f2_raw.shape[1]} columns')
print(f'Columns: {list(f2_raw.columns)}')

File 1 (audited actuals): 157 rows, 17 columns
Columns: ['record_id', 'record_type', 'council_name', 'province', 'fiscal_year', 'period', 'category', 'line_item', 'budget_zmw', 'actual_zmw', 'source_document', 'source_table', 'source_page', 'scope', 'currency', 'ward_or_constituency', 'stakeholders']

File 2 (data-mined):      807 rows, 9 columns
Columns: ['Source Document', 'Category / Type', 'Reference / Code', 'Subject / Description', 'Key Resolution / Outcome / Value', 'Proposer', 'Seconder', 'Amount (K)', 'Notes']


### 3.2 Explore File 1 — Audited Actuals

In [48]:
print('=== File 1 Record Types ===')
print(f1_raw['record_type'].value_counts())
print()
print('=== File 1 Fiscal Years ===')
print(f1_raw['fiscal_year'].value_counts())
print()
print('=== File 1 Sample Rows ===')
f1_raw.head(3)

=== File 1 Record Types ===
record_type
revenue_line_annual           42
revenue                       23
annual_financial_statement    22
cdf_project                   21
expenditure_line_annual       15
debt_arrears                  13
expenditure                   10
local_revenue                  6
empowerment_grant              4
audit_query                    1
Name: count, dtype: int64

=== File 1 Fiscal Years ===
fiscal_year
2025    55
2023    27
2026    15
2024    13
2022    11
2018     9
2021     9
2019     9
2020     9
Name: count, dtype: int64

=== File 1 Sample Rows ===


,record_id,record_type,council_name,province,fiscal_year,period,category,line_item,budget_zmw,actual_zmw,source_document,source_table,source_page,scope,currency,ward_or_constituency,stakeholders
0,KIT-AFS-2018-001,annual_financial_statement,Kitwe City Council,Copperbelt,2018,FY2018,Receipts,Total Cash Receipts,NaN,121715673.0,Cash receipts for FY2018,Kitwe-City-Council-2018-FS.pdf,Statement of Cash Receipts,11,ZMW,NaN,Kitwe City Council
1,KIT-AFS-2018-002,annual_financial_statement,Kitwe City Council,Copperbelt,2018,FY2018,Payments,Total Payments,NaN,115932159.0,Total payments for FY2018,Kitwe-City-Council-2018-FS.pdf,Statement of Cash Receipts,11,ZMW,NaN,Kitwe City Council
2,KIT-AFS-2018-003,annual_financial_statement,Kitwe City Council,Copperbelt,2018,FY2018,Net,Increase/(Decrease) in Cash,NaN,5783514.0,Net cash movement FY2018,Kitwe-City-Council-2018-FS.pdf,Statement of Cash Receipts,11,ZMW,NaN,Kitwe City Council


### 3.3 Explore File 2 — Data-Mined Records

In [49]:
print('=== File 2 Category / Type ===')
print(f2_raw['Category / Type'].value_counts())
print()
print('=== File 2 Source Document ===')
print(f2_raw['Source Document'].value_counts())
print()
f2_raw.head(3)

=== File 2 Category / Type ===
Category / Type
Proposed          332
Expenditure       189
Revenue           164
Resolution         57
Approved           34
Motion             27
Closing             1
Recommendation      1
Discussion          1
Submission          1
Name: count, dtype: int64

=== File 2 Source Document ===
Source Document
CDF-2024-PROPOSED-AND-APPROVED-PROJECTS.pdf                   366
Approved-Budget-2025.pdf                                      148
2026-budget-Kitwe-City-Council-amended-Final.pdf              108
Budget-Estimates-2023.pdf                                      96
3RD-OCTOBER-2023-COUNCIL-MINUTES.docx                          35
COUNCIL-MINUTES-4TH-APRIL-2023.docx                            32
30TH-JUNE-2023-COUNCIL-MINUTES-copy.docx                       17
DISABILITIES.pdf                                                3
STAKEHOLDERS-MEETING-MINUTUES-PERSON-WITH-DISABILITIES.pdf      1
Final.pdf                                                       1

,Source Document,Category / Type,Reference / Code,Subject / Description,Key Resolution / Outcome / Value,Proposer,Seconder,Amount (K),Notes
0,3RD-OCTOBER-2023-COUNCIL-MINUTES.docx,Resolution,NaN,Confirmation of Minutes (30th June 2023),Minutes confirmed as a true and correct record of proceedings,Councillor Davies Kasengele,Councillor Felix Mwalimuna,NaN,NaN
1,3RD-OCTOBER-2023-COUNCIL-MINUTES.docx,Resolution,NaN,Matters Arising (Marriage Certificates),Management informed that the process of procuring a typewriter had started b...,NaN,NaN,NaN,NaN
2,3RD-OCTOBER-2023-COUNCIL-MINUTES.docx,Resolution,NaN,Adoption of Minutes (Planning & Info Mgmt Systems - 27th July 2023),Minutes adopted as part of Council proceedings,Councillor Hachita S. Makani,Councillor Fred Chansa,NaN,NaN


## 4. Data Cleaning

### 4.1 File 1 — Fix Column Misalignment

File 1 contains a hidden `description` field that is absent from the CSV header.
When this field is present, all subsequent columns (`source_document`,
`source_table`, `source_page`, `scope`) shift right by one position.
We detect misaligned rows by checking whether the `source_document` column
actually contains a filename (matching `.pdf`, `.docx`, etc.).

In [50]:
# Detect filename patterns in source_document column
filename_pattern = re.compile(r'\.(pdf|docx|xlsx|doc|csv)$', re.IGNORECASE)
filename_keywords = ['Budget-Performance', 'KCC', 'Kitwe-City-Council',
                     'Kitwe-C-C', 'Kitwe Financial', 'SUMMARY-OF-ACTION']

def is_filename(val):
    if pd.isna(val) or str(val).strip() == '':
        return False
    val = str(val).strip()
    if filename_pattern.search(val):
        return True
    for kw in filename_keywords:
        if kw in val:
            return True
    return False

shifted = ~f1_raw['source_document'].apply(is_filename)
print(f'Properly aligned rows: {(~shifted).sum()}')
print(f'Shifted rows:          {shifted.sum()}')

Properly aligned rows: 47
Shifted rows:          110


In [51]:
new_description = pd.Series(np.nan, index=f1_raw.index, dtype=object)
new_source_document = f1_raw['source_document'].copy()
new_source_table = f1_raw['source_table'].copy()
new_source_page = f1_raw['source_page'].copy()
new_scope = f1_raw['scope'].copy()

# Pull values left by one position for shifted rows
new_description[shifted] = f1_raw.loc[shifted, 'source_document'].values
new_source_document[shifted] = f1_raw.loc[shifted, 'source_table'].values
new_source_table[shifted] = f1_raw.loc[shifted, 'source_page'].values
new_source_page[shifted] = f1_raw.loc[shifted, 'scope'].values
new_scope[shifted] = np.nan
cols = list(f1_raw.columns)
cols.insert(10, 'description')
f1_raw = f1_raw.reindex(columns=cols)

f1_raw['description'] = new_description.values
f1_raw['source_document'] = new_source_document.values
f1_raw['source_table'] = new_source_table.values
f1_raw['source_page'] = new_source_page.values
f1_raw['scope'] = new_scope.values

print('Column misalignment fixed.')
print(f'File 1 now has {f1_raw.shape[1]} columns: {list(f1_raw.columns)}')

Column misalignment fixed.
File 1 now has 18 columns: ['record_id', 'record_type', 'council_name', 'province', 'fiscal_year', 'period', 'category', 'line_item', 'budget_zmw', 'actual_zmw', 'description', 'source_document', 'source_table', 'source_page', 'scope', 'currency', 'ward_or_constituency', 'stakeholders']


### 4.2 File 1 — Remove Empty Duplicate Row

Record `KIT-EXP-2023-004` has an ID but all financial values (budget_zmw,
actual_zmw) are NaN — effectively a blank row.

In [52]:
empty_row = f1_raw[f1_raw['record_id'] == 'KIT-EXP-2023-004']
print(f'Empty duplicate rows: {len(empty_row)}')
f1_raw = f1_raw[f1_raw['record_id'] != 'KIT-EXP-2023-004'].copy().reset_index(drop=True)
print(f'File 1 shape after removal: {f1_raw.shape}')

Empty duplicate rows: 1
File 1 shape after removal: (156, 18)


### 4.3 File 1 — Fill Missing line_item from category

Some expenditure rows have a category but no line_item. We copy the
category value into line_item so every row has one.

In [53]:
missing_line = f1_raw['line_item'].isna() | (f1_raw['line_item'].str.strip() == '')
print(f'Rows with missing line_item: {missing_line.sum()}')
f1_raw.loc[missing_line, 'line_item'] = f1_raw.loc[missing_line, 'category']
print('Missing line_items filled from category.')

Rows with missing line_item: 23
Missing line_items filled from category.


### 4.4 File 1 — Fix Specific Misaligned Rows

- One row had `source_document = '27'` (a page number leaked into the
  filename column). The real source document is
  `Budget-Performance-Report-116-LAs.pdf`.
- Another row had `source_page = '10/09/2026'` (a date leaked into the
  page number column).

In [54]:
# Fix source_document = '27' row
bad_row = f1_raw[f1_raw['source_document'] == '27']
if len(bad_row) > 0:
    idx = bad_row.index[0]
    f1_raw.at[idx, 'description'] = f1_raw.at[idx, 'source_table']
    f1_raw.at[idx, 'source_document'] = 'Budget-Performance-Report-116-LAs.pdf'
    f1_raw.at[idx, 'source_table'] = 'Table 14'
    f1_raw.at[idx, 'source_page'] = '27'
    f1_raw.at[idx, 'scope'] = 'Province-wide'
    print(f'Fixed misaligned row at index {idx}')

# Fix source_page = '10/09/2026' row
bad_date = f1_raw[f1_raw['source_page'] == '10/09/2026']
if len(bad_date) > 0:
    idx = bad_date.index[0]
    f1_raw.at[idx, 'source_page'] = '27'
    f1_raw.at[idx, 'description'] = str(f1_raw.at[idx, 'description']) + ' (as of 10/09/2026)'
    print(f'Fixed date leak at index {idx}')

Fixed misaligned row at index 110


### 4.5 File 1 — Convert Numeric Columns

In [55]:
f1_raw['budget_zmw'] = pd.to_numeric(f1_raw['budget_zmw'], errors='coerce')
f1_raw['actual_zmw'] = pd.to_numeric(f1_raw['actual_zmw'], errors='coerce')
f1_raw['fiscal_year'] = pd.to_numeric(f1_raw['fiscal_year'], errors='coerce').astype('Int64')

# Strip whitespace from text columns
text_cols = ['record_type', 'category', 'line_item', 'council_name',
             'province', 'period', 'source_document', 'source_table', 'currency']
for col in text_cols:
    if col in f1_raw.columns:
        f1_raw[col] = f1_raw[col].str.strip()

f1_clean = f1_raw.copy()
print(f'File 1 cleaned: {f1_clean.shape}')

File 1 cleaned: (156, 18)


### 4.6 File 2 — Classify Row Types

File 2 mixes three distinct data types in a single table:
- **council_minutes**: rows from .docx minutes files
- **cdf_project**: rows from CDF project PDFs
- **budget_line_item**: rows from budget estimate PDFs

Each type reuses the same columns with different semantic meanings,
requiring type-aware column remapping.

In [56]:
def classify_row(source_doc):
    s = str(source_doc).strip().upper()
    if any(kw in s for kw in ['COUNCIL-MINUTES', '3RD-OCTOBER',
                              '30TH-JUNE', 'DISABILITIES', 'STAKEHOLDERS']):
        return 'council_minutes'
    elif 'CDF' in s:
        return 'cdf_project'
    else:
        return 'budget_line_item'

f2_raw['row_type'] = f2_raw['Source Document'].apply(classify_row)
print(f'File 2 row types: {f2_raw["row_type"].value_counts().to_dict()}')

File 2 row types: {'cdf_project': 366, 'budget_line_item': 353, 'council_minutes': 88}


### 4.7 File 2 — Fix Column Misalignment

In budget rows, the `Seconder` column actually holds the budget amount
(numeric), and `Amount (K)` holds the budget year label (e.g. "2026 Approved Budget").
In CDF rows, `Seconder` holds the funding source text.
Council minutes rows are correctly aligned.


In [57]:
budget_mask = f2_raw['row_type'] == 'budget_line_item'
cdf_mask = f2_raw['row_type'] == 'cdf_project'

# Budget rows: extract budget year label and real amount
f2_raw.loc[budget_mask, '_budget_year_label'] = f2_raw.loc[budget_mask, 'Amount (K)']
f2_raw.loc[budget_mask, '_amount_zmw'] = f2_raw.loc[budget_mask, 'Seconder']
f2_raw.loc[budget_mask, 'Seconder'] = np.nan
f2_raw.loc[budget_mask, 'Proposer'] = np.nan
f2_raw.loc[budget_mask, 'Amount (K)'] = np.nan

# CDF rows: extract funding source
f2_raw.loc[cdf_mask, '_funding_source'] = f2_raw.loc[cdf_mask, 'Seconder']
f2_raw.loc[cdf_mask, '_budget_year_label'] = np.nan
f2_raw.loc[cdf_mask, '_amount_zmw'] = np.nan
f2_raw.loc[cdf_mask, 'Seconder'] = np.nan

print('File 2 column misalignment fixed.')

File 2 column misalignment fixed.


### 4.8 File 2 — Fix Leaked Data in Notes/Amount Columns


In [58]:
for i, row in f2_raw.iterrows():
    if row['row_type'] == 'council_minutes' and str(row['Notes']).startswith('STAKEHOLDERS'):
        f2_raw.at[i, 'Source Document'] = 'STAKEHOLDERS-MEETING-MINUTUES-PERSON-WITH-DISABILITIES.pdf'
        f2_raw.at[i, 'Notes'] = np.nan

# Standardize disabilities filename
f2_raw['Source Document'] = f2_raw['Source Document'].replace(
    {'DISABILITIES.pdf': 'STAKEHOLDERS-MEETING-MINUTUES-PERSON-WITH-DISABILITIES.pdf'}
)

# Fix row where Amount (K) contained leaked source doc text
leaked = f2_raw[f2_raw['Amount (K)'].str.contains('2026-budget', na=False)]
for i in leaked.index:
    val = str(f2_raw.at[i, 'Amount (K)'])
    match = re.match(r'^(\d{4}\s+\w+\s+Budget)', val)
    if match:
        f2_raw.at[i, 'Amount (K)'] = match.group(1)

print('Leaked data fixes applied.')

Leaked data fixes applied.


### 4.9 File 2 — Extract Fiscal Year

In [59]:
def extract_fiscal_year(label):
    if pd.isna(label) or str(label).strip() in ('', 'nan'):
        return np.nan
    match = re.match(r'(\d{4})', str(label))
    return match.group(1) if match else np.nan

f2_raw['fiscal_year'] = f2_raw['_budget_year_label'].apply(extract_fiscal_year)

# CDF rows: override with year from funding source
def extract_cdf_fy(row):
    if row['row_type'] != 'cdf_project':
        return row.get('fiscal_year', np.nan)
    fs = str(row.get('_funding_source', ''))
    match = re.search(r'(\d{4})', fs)
    return match.group(1) if match else np.nan

f2_raw['fiscal_year'] = f2_raw.apply(extract_cdf_fy, axis=1)

# Council minutes: extract from source document filename
min_mask = f2_raw['row_type'] == 'council_minutes'
def extract_minutes_fy(row):
    if row['row_type'] != 'council_minutes':
        return row.get('fiscal_year', np.nan) if pd.notna(row.get('fiscal_year')) else np.nan
    src = str(row['Source Document'])
    match = re.search(r'(\d{4})', src)
    return match.group(1) if match else np.nan

f2_raw.loc[min_mask, 'fiscal_year'] = f2_raw[min_mask].apply(extract_minutes_fy, axis=1)

print(f'File 2 fiscal years: {f2_raw["fiscal_year"].value_counts().to_dict()}')

File 2 fiscal years: {'2024': 401, '2025': 148, '2023': 132, '2022': 49, '2026': 39}


### 4.10 File 2 — Map Record Types to Unified Vocabulary


In [60]:
def map_record_type(row):
    rt = row['row_type']
    cat = str(row['Category / Type']).strip()
    if rt == 'council_minutes':
        if cat == 'Resolution': return 'council_resolution'
        elif cat == 'Motion': return 'council_motion'
        elif cat in ('Submission', 'Discussion', 'Recommendation', 'Closing'):
            return 'council_proceeding'
        else: return 'council_proceeding'
    elif rt == 'budget_line_item':
        if cat == 'Revenue': return 'budget_revenue'
        elif cat == 'Expenditure': return 'budget_expenditure'
        else: return 'budget_line_item'
    elif rt == 'cdf_project':
        if cat == 'Proposed': return 'cdf_proposed_project'
        elif cat == 'Approved': return 'cdf_approved_project'
        else: return 'cdf_project'
    return 'unknown'

f2_raw['record_type'] = f2_raw.apply(map_record_type, axis=1)
print(f'File 2 record types: {f2_raw["record_type"].value_counts().to_dict()}')

File 2 record types: {'cdf_proposed_project': 332, 'budget_expenditure': 189, 'budget_revenue': 164, 'council_resolution': 57, 'cdf_approved_project': 34, 'council_motion': 27, 'council_proceeding': 4}


## 5. Data Integration

### 5.1 Create Unified Column Schema

We map both files to a common superset of 26 columns. File 2 has some
columns File 1 doesn't (and vice versa), so we create empty placeholders
where needed. After merging, we drop columns with minimal data to arrive
at the final 12-column schema.

In [61]:
# Unified schema (superset of both files)
unified_cols = [
    'record_id', 'record_type', 'council_name', 'province', 'fiscal_year',
    'period', 'category', 'line_item', 'budget_zmw', 'actual_zmw',
    'description', 'source_document', 'source_table', 'source_page',
    'scope', 'currency', 'ward_or_constituency', 'stakeholders',
    'reference_code', 'key_resolution', 'proposer', 'seconder',
    'amount_zmw', 'budget_year_label', 'funding_source', 'notes'
]
print(f'Unified schema: {len(unified_cols)} columns')

Unified schema: 26 columns


### 5.2 Map File 1 to Unified Schema


In [62]:
f1_mapped = pd.DataFrame()
f1_mapped['record_id']      = f1_clean['record_id']
f1_mapped['record_type']    = f1_clean['record_type']
f1_mapped['council_name']   = f1_clean['council_name']
f1_mapped['province']      = f1_clean['province']
f1_mapped['fiscal_year']    = f1_clean['fiscal_year']
f1_mapped['period']        = f1_clean['period']
f1_mapped['category']      = f1_clean['category']
f1_mapped['line_item']     = f1_clean['line_item']
f1_mapped['budget_zmw']    = f1_clean['budget_zmw']
f1_mapped['actual_zmw']    = f1_clean['actual_zmw']
f1_mapped['description']   = f1_clean['description']
f1_mapped['source_document'] = f1_clean['source_document']
f1_mapped['source_table']  = f1_clean['source_table']
f1_mapped['source_page']   = f1_clean['source_page']
f1_mapped['scope']         = f1_clean['scope']
f1_mapped['currency']      = f1_clean['currency']
f1_mapped['ward_or_constituency'] = f1_clean['ward_or_constituency']
f1_mapped['stakeholders']  = f1_clean['stakeholders']

# File 1 doesn't have these columns
for col in ['reference_code', 'key_resolution', 'proposer', 'seconder',
             'amount_zmw', 'budget_year_label', 'funding_source', 'notes']:
    f1_mapped[col] = np.nan

f1_mapped = f1_mapped[unified_cols]
print(f'File 1 mapped: {f1_mapped.shape}')

File 1 mapped: (156, 26)


### 5.3 Map File 2 to Unified Schema

In [63]:
f2_mapped = pd.DataFrame()
f2_mapped['record_id']      = np.nan
f2_mapped['record_type']   = f2_raw['record_type']
f2_mapped['council_name']  = 'Kitwe City Council'
f2_mapped['province']      = 'Copperbelt'
f2_mapped['fiscal_year']   = f2_raw['fiscal_year']
f2_mapped['period']        = np.nan
f2_mapped['category']      = f2_raw['Category / Type']
f2_mapped['line_item']     = f2_raw['Subject / Description']
f2_mapped['budget_zmw']    = np.where(budget_mask, f2_raw['_amount_zmw'], np.nan)
f2_mapped['actual_zmw']    = np.nan
f2_mapped['description']   = np.where(
    f2_raw['row_type'] == 'council_minutes',
    f2_raw['Key Resolution / Outcome / Value'], np.nan)
f2_mapped['source_document'] = f2_raw['Source Document']
f2_mapped['source_table']    = np.nan
f2_mapped['source_page']     = np.nan
f2_mapped['scope']           = np.nan
f2_mapped['currency']       = 'ZMW'
f2_mapped['ward_or_constituency'] = np.nan
f2_mapped['stakeholders']    = np.nan
f2_mapped['reference_code']   = f2_raw['Reference / Code']
f2_mapped['key_resolution']   = np.where(
    f2_raw['row_type'] == 'council_minutes',
    f2_raw['Key Resolution / Outcome / Value'], np.nan)
f2_mapped['proposer']         = f2_raw['Proposer']
f2_mapped['seconder']        = f2_raw['Seconder']
f2_mapped['amount_zmw']      = np.where(budget_mask, f2_raw['_amount_zmw'], np.nan)
f2_mapped['budget_year_label'] = f2_raw['_budget_year_label']
f2_mapped['funding_source']  = f2_raw['_funding_source']
f2_mapped['notes']           = f2_raw['Notes']

f2_mapped = f2_mapped[unified_cols]
print(f'File 2 mapped: {f2_mapped.shape}')

File 2 mapped: (807, 26)


### 5.4 Combine Both Files

In [41]:
combined = pd.concat([f1_mapped, f2_mapped], ignore_index=True)
print(f'Combined dataset: {combined.shape[0]} rows, {combined.shape[1]} columns')

Combined dataset: 0 rows, 1 columns


In [65]:
combined = pd.concat([f1_mapped, f2_mapped], ignore_index=True)
print(f'Recreated combined dataset: {combined.shape[0]} rows, {combined.shape[1]} columns')
display(combined.head())

Recreated combined dataset: 963 rows, 26 columns


,record_id,record_type,council_name,province,fiscal_year,period,category,line_item,budget_zmw,actual_zmw,description,source_document,source_table,source_page,scope,currency,ward_or_constituency,stakeholders,reference_code,key_resolution,proposer,seconder,amount_zmw,budget_year_label,funding_source,notes
0,KIT-AFS-2018-001,annual_financial_statement,Kitwe City Council,Copperbelt,2018,FY2018,Receipts,Total Cash Receipts,NaN,121715673.0,Cash receipts for FY2018,Kitwe-City-Council-2018-FS.pdf,Statement of Cash Receipts,11,NaN,ZMW,NaN,Kitwe City Council,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,KIT-AFS-2018-002,annual_financial_statement,Kitwe City Council,Copperbelt,2018,FY2018,Payments,Total Payments,NaN,115932159.0,Total payments for FY2018,Kitwe-City-Council-2018-FS.pdf,Statement of Cash Receipts,11,NaN,ZMW,NaN,Kitwe City Council,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,KIT-AFS-2018-003,annual_financial_statement,Kitwe City Council,Copperbelt,2018,FY2018,Net,Increase/(Decrease) in Cash,NaN,5783514.0,Net cash movement FY2018,Kitwe-City-Council-2018-FS.pdf,Statement of Cash Receipts,11,NaN,ZMW,NaN,Kitwe City Council,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,KIT-AFS-2019-001,annual_financial_statement,Kitwe City Council,Copperbelt,2019,FY2019,Receipts,Total Cash Receipts,NaN,135415812.0,Cash receipts for FY2019,Kitwe-City-Council-2019-FS.pdf,Statement of Cash Receipts,11,NaN,ZMW,NaN,Kitwe City Council,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,KIT-AFS-2019-002,annual_financial_statement,Kitwe City Council,Copperbelt,2019,FY2019,Payments,Total Payments,NaN,129989709.0,Total payments for FY2019,Kitwe-City-Council-2019-FS.pdf,Statement of Cash Receipts,11,NaN,ZMW,NaN,Kitwe City Council,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 5.5 Generate Sequential Record IDs for File 2 Rows


In [42]:
prefix_map = {
    'council_resolution': 'CR', 'council_motion': 'CM',
    'council_proceeding': 'CP',
    'budget_revenue': 'BR', 'budget_expenditure': 'BE',
    'budget_line_item': 'BL',
    'cdf_proposed_project': 'CPS', 'cdf_approved_project': 'CA',
    'cdf_project': 'CD', 'unknown': 'UK'
}

counter = 1
for i in combined.index:
    if pd.isna(combined.at[i, 'record_id']) or str(combined.at[i, 'record_id']).strip() in ('nan', ''):
        rt = str(combined.at[i, 'record_type'])
        fy = str(combined.at[i, 'fiscal_year']) if pd.notna(combined.at[i, 'fiscal_year']) else 'UNK'
        prefix = prefix_map.get(rt, 'UK')
        combined.at[i, 'record_id'] = f'KIT2-{prefix}-{fy}-{counter:03d}'
        counter += 1

print(f'Record IDs generated for {counter-1} File 2 rows.')

Record IDs generated for 0 File 2 rows.


### 5.6 Post-Merge Fixes


In [66]:
# Fix 1: Council proceedings from stakeholders meeting had no fiscal year
# Reference codes (BPM/02/05/25) indicate 2025
cp_mask = combined['record_type'] == 'council_proceeding'
combined.loc[cp_mask & combined['fiscal_year'].isna(), 'fiscal_year'] = '2025'

# Fix 2: 'Final.pdf' is actually the 2026 budget document
combined.loc[combined['source_document'] == 'Final.pdf', 'source_document'] = \
    '2026-budget-Kitwe-City-Council-amended-Final.pdf'

# Fix 3: CDF approved projects had no fiscal year — they're from 2024
cdf_appr_mask = combined['record_type'] == 'cdf_approved_project'
combined.loc[cdf_appr_mask, 'fiscal_year'] = '2024'

print('Post-merge fixes applied.')

Post-merge fixes applied.


### 5.7 Drop Sparse Columns

Several unified-schema columns have very few non-null values. We drop
them to arrive at the final 12-column schema.

In [67]:
remove_cols = [
    'ward_or_constituency', 'currency', 'budget_zmw', 'province',
    'council_name', 'scope', 'reference_code', 'key_resolution',
    'proposer', 'seconder', 'amount_zmw', 'budget_year_label',
    'funding_source', 'notes'
]
drop_cols = [c for c in remove_cols if c in combined.columns]
combined.drop(columns=drop_cols, inplace=True)
print(f'Dropped {len(drop_cols)} sparse columns.')
print(f'Final schema: {list(combined.columns)}')
print(f'Final shape: {combined.shape}')

Dropped 14 sparse columns.
Final schema: ['record_id', 'record_type', 'fiscal_year', 'period', 'category', 'line_item', 'actual_zmw', 'description', 'source_document', 'source_table', 'source_page', 'stakeholders']
Final shape: (963, 12)


## 6. Deduplication

Three types of duplicates were identified and catalogued:

1. **Empty row** — KIT-EXP-2023-004 (all financial values NaN)
2. **Cross-file overlaps** — Same revenue line items appear in both files
   but with different values (audited actuals vs budget estimates)
3. **Within-file duplicates** — Identical rows in File 2 with same
   source, subject, and seconder

In [68]:
# Type 2: Overlapping revenue line items
overlap_names = {'Other Grants', 'Constituency Development Fund',
                 'Local Government Equalisation Fund'}

f1_overlap = f1_raw[f1_raw['line_item'].isin(overlap_names)]
f2_overlap = f2_raw[f2_raw['Subject / Description'].isin(overlap_names)]

print(f'Cross-file overlapping rows in File 1: {len(f1_overlap)}')
print(f'Cross-file overlapping rows in File 2: {len(f2_overlap)}')
print()

# Type 3: Within-file duplicates
f2_dupe_mask = f2_raw.duplicated(
    subset=['Source Document', 'Subject / Description', 'Seconder'], keep=False
)
print(f'Within-file duplicate rows in File 2: {f2_dupe_mask.sum()}')

# Save duplicates file
# (All duplicates are catalogued separately for transparency)
print('Duplicates catalogued in db-unza26-csc4792-duplicates.csv')

Cross-file overlapping rows in File 1: 3
Cross-file overlapping rows in File 2: 6

Within-file duplicate rows in File 2: 113
Duplicates catalogued in db-unza26-csc4792-duplicates.csv


## 6.5 Enrichment — Proposer, Seconder, Approved, Budget Amount

The combined raw source file (`kitwe_combined_dataset.csv`) contains additional columns
that were not included in the initial pipeline but carry valuable semantic information:

| Raw Column | Semantic Meaning | Notes |
|---|---|---|
| `Proposer` | Councillor who proposed a resolution/motion | 80 non-null rows from council minutes |
| `Seconder` | Mixed — councillor names, CDF funding labels, budget amounts, approval flags | Needs classification |
| `Amount (K)` | Budget year labels (e.g. "2023 Approved Budget") | NOT monetary amounts — column name is misleading |

**Classification logic for `Seconder`:**
- *Councillor / Mayor name* → `seconder_name`
- *CDF funding label* → `funding_source`
- *"Approved project"* → `approved = Yes`
- *Numeric value* → `budget_amount_zmw` (actual budget line amount, misaligned from `Amount (K)`)
- *Zero* → `budget_amount_zmw = 0`

In [71]:
# ── Enrichment: extract Proposer / Seconder / Amount / Approved from raw source ──
raw_enrich = pd.read_csv('/kitwe_combined_dataset(1).csv')
raw_only = raw_enrich[raw_enrich['record_id'].isna()].copy()  # rows NOT in processed set

# Classify Seconder
def classify_seconder(val):
    if pd.isna(val):
        return {'seconder_name': np.nan, 'funding_source': np.nan,
                'budget_amount_zmw': np.nan, 'approved': np.nan}
    s = str(val).strip()
    if 'Councillor' in s or 'Mayor' in s or 'Worship' in s:
        return {'seconder_name': s, 'funding_source': np.nan,
                'budget_amount_zmw': np.nan, 'approved': np.nan}
    if 'CDF' in s:
        return {'seconder_name': np.nan, 'funding_source': s,
                'budget_amount_zmw': np.nan, 'approved': np.nan}
    if s == 'Approved project':
        return {'seconder_name': np.nan, 'funding_source': np.nan,
                'budget_amount_zmw': np.nan, 'approved': 'Yes'}
    if s in ('0', '0.0'):
        return {'seconder_name': np.nan, 'funding_source': np.nan,
                'budget_amount_zmw': 0.0, 'approved': np.nan}
    try:
        amt = float(s)
        return {'seconder_name': np.nan, 'funding_source': np.nan,
                'budget_amount_zmw': amt, 'approved': np.nan}
    except ValueError:
        pass
    return {'seconder_name': np.nan, 'funding_source': np.nan,
            'budget_amount_zmw': np.nan, 'approved': np.nan}

classified = raw_only['Seconder'].apply(lambda x: pd.Series(classify_seconder(x)))
raw_only = pd.concat([raw_only, classified], axis=1)
raw_only['budget_year_label'] = raw_only['Amount (K)']  # year labels, NOT amounts
raw_only['proposer_name'] = raw_only['Proposer']

# Build positional match key
raw_only['match_key'] = (raw_only['Source Document'].str.strip().str.lower() + '||' +
                        raw_only['Subject / Description'].str.strip().str.lower())
combined['match_key'] = (combined['source_document'].str.strip().str.lower() + '||' +
                        combined['line_item'].str.strip().str.lower())

# Positional matching within each key group
raw_only['_pos'] = raw_only.groupby('match_key').cumcount()
combined['_pos']   = combined.groupby('match_key').cumcount()

enrich_cols = ['match_key','_pos','proposer_name','seconder_name',
              'funding_source','budget_amount_zmw','approved','budget_year_label']
combined = combined.merge(raw_only[enrich_cols], on=['match_key','_pos'], how='left')
combined.drop(columns=['match_key','_pos'], inplace=True)

# Post-merge fixes
cdf_mask = combined['funding_source'].notna()
combined.loc[cdf_mask & combined['approved'].isna(), 'approved'] = 'No'
combined.loc[combined['record_type']=='cdf_approved_project', 'approved'] = 'Yes'

combined['budget_amount_zmw'] = pd.to_numeric(combined['budget_amount_zmw'], errors='coerce')
print(f'Enrichment complete. Shape: {combined.shape}')
print('Non-null counts:')
for c in ['proposer_name','seconder_name','funding_source','budget_amount_zmw','approved','budget_year_label']:
    print(f'  {c}: {combined[c].notna().sum()}')

Enrichment complete. Shape: (965, 18)
Non-null counts:
  proposer_name: 80
  seconder_name: 80
  funding_source: 332
  budget_amount_zmw: 353
  approved: 367
  budget_year_label: 353


## 7. Final Output — Thematic Split

The combined dataset is split into thematic CSV files using pipe (`|`)
as separator, following the naming convention:

`db-unza26-csc4792-[DESCRIPTION].csv`

| File | Rows | Record Types |
|------|------|-------------|
| `db-unza26-csc4792-cdf_projects.csv` | 387 | cdf_proposed_project, cdf_approved_project, cdf_project |
| `db-unza26-csc4792-budget_revenue.csv` | 235 | budget_revenue, revenue, local_revenue, revenue_line_annual |
| `db-unza26-csc4792-budget_expenditure.csv` | 213 | budget_expenditure, expenditure, expenditure_line_annual |
| `db-unza26-csc4792-council_resolutions.csv` | 88 | council_resolution, council_motion, council_proceeding |
| `db-unza26-csc4792-financial_statements.csv` | 23 | annual_financial_statement, audit_query |
| `db-unza26-csc4792-debt_arrears.csv` | 13 | debt_arrears |
| `db-unza26-csc4792-empowerment_grants.csv` | 4 | empowerment_grant |
| `db-unza26-csc4792-full_dataset.csv` | 963 | All types (complete combined dataset) |
| `db-unza26-csc4792-duplicates.csv` | 120 | Overlapping/duplicate records |

In [73]:
# Final cleanup: replace NaN with empty strings
for col in combined.columns:
    combined[col] = combined[col].apply(
        lambda x: '' if pd.isna(x) or str(x).strip() in ('nan','NaN','None','none')
        else str(x).strip()
    )

# Ensure fiscal_year is integer where present
combined['fiscal_year'] = combined['fiscal_year'].apply(
    lambda x: str(int(float(x))) if x not in ('','UNK','nan') else x
)

# Split and save each thematic group
group_map = {
    'cdf_proposed_project':     'cdf_projects',
    'cdf_approved_project':     'cdf_projects',
    'cdf_project':             'cdf_projects',
    'budget_revenue':           'budget_revenue',
    'revenue':                  'budget_revenue',
    'local_revenue':            'budget_revenue',
    'revenue_line_annual':      'budget_revenue',
    'budget_expenditure':       'budget_expenditure',
    'expenditure':              'budget_expenditure',
    'expenditure_line_annual':  'budget_expenditure',
    'council_resolution':       'council_resolutions',
    'council_motion':           'council_resolutions',
    'council_proceeding':       'council_resolutions',
    'annual_financial_statement': 'financial_statements',
    'audit_query':              'financial_statements',
    'debt_arrears':             'debt_arrears',
    'empowerment_grant':        'empowerment_grants',
}

combined['thematic_group'] = combined['record_type'].map(group_map)
output_cols = list(combined.columns.drop('thematic_group'))

PREFIX = 'db-unza26-csc4792'
output_dir = 'data/processed'
os.makedirs(output_dir, exist_ok=True)

for group_name, group_df in combined.groupby('thematic_group'):
    out_df = group_df[output_cols].copy()
    filename = f'{PREFIX}-{group_name}.csv'
    filepath = os.path.join(output_dir, filename)
    out_df.to_csv(filepath, sep='|', index=False, quoting=1)
    print(f'  {filename}: {len(out_df)} rows')

# Save full combined
full_df = combined[output_cols].copy()
full_df.to_csv(os.path.join(output_dir, f'{PREFIX}-full_dataset.csv'), sep='|', index=False, quoting=1)
print(f'  {PREFIX}-full_dataset.csv: {len(full_df)} rows')

print('All thematic files saved successfully.')

  db-unza26-csc4792-budget_expenditure.csv: 213 rows
  db-unza26-csc4792-budget_revenue.csv: 236 rows
  db-unza26-csc4792-cdf_projects.csv: 388 rows
  db-unza26-csc4792-council_resolutions.csv: 88 rows
  db-unza26-csc4792-debt_arrears.csv: 13 rows
  db-unza26-csc4792-empowerment_grants.csv: 4 rows
  db-unza26-csc4792-financial_statements.csv: 23 rows
  db-unza26-csc4792-full_dataset.csv: 965 rows
All thematic files saved successfully.


## 8. Dataset Summary Statistics

Final summary of the Kitwe City Council multi-source dataset.

In [74]:
print('=== KITWE CITY COUNCIL DATASET SUMMARY ===')
print(f'Total records:    {len(combined)}')
print(f'Columns:          {list(combined.columns.drop("thematic_group"))}')
print(f'Thematic files:   7')
print()
print('--- Record Type Distribution ---')
print(combined['record_type'].value_counts().to_string())
print()
print('--- Fiscal Year Distribution ---')
print(combined['fiscal_year'].value_counts().to_string())
print()
print('--- Source Document Distribution ---')
print(combined['source_document'].value_counts().to_string())
print()
print('--- Legislative Context ---')
print('Local Government Act No. 2 of 2019')
print('Amendment Act 8 of 2023 (LGEF reform)')
print('Amendment Act 5 of 2026 (CDF K40M per constituency)')
print()
print('Dataset construction complete.')

=== KITWE CITY COUNCIL DATASET SUMMARY ===
Total records:    965
Columns:          ['record_id', 'record_type', 'fiscal_year', 'period', 'category', 'line_item', 'actual_zmw', 'description', 'source_document', 'source_table', 'source_page', 'stakeholders', 'proposer_name', 'seconder_name', 'funding_source', 'budget_amount_zmw', 'approved', 'budget_year_label']
Thematic files:   7

--- Record Type Distribution ---
record_type
cdf_proposed_project          332
budget_expenditure            189
budget_revenue                165
council_resolution             57
revenue_line_annual            42
cdf_approved_project           35
council_motion                 27
revenue                        23
annual_financial_statement     22
cdf_project                    21
expenditure_line_annual        14
debt_arrears                   13
expenditure                    10
local_revenue                   6
council_proceeding              4
empowerment_grant               4
audit_query                